<a href="https://colab.research.google.com/github/amitdey7/MS107/blob/master/LFM_VL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git@3c2517727ce28a30f5044e01663ee204deb1cdbe pillow

  Cloning https://github.com/huggingface/transformers.git (to revision 3c2517727ce28a30f5044e01663ee204deb1cdbe) to /tmp/pip-req-build-u30z3qpk
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-u30z3qpk
  Running command git rev-parse -q --verify 'sha^3c2517727ce28a30f5044e01663ee204deb1cdbe'
  Running command git fetch -q https://github.com/huggingface/transformers.git 3c2517727ce28a30f5044e01663ee204deb1cdbe
  Running command git checkout -q 3c2517727ce28a30f5044e01663ee204deb1cdbe
  Resolved https://github.com/huggingface/transformers.git to commit 3c2517727ce28a30f5044e01663ee204deb1cdbe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image

# Load model and processor
model_id = "LiquidAI/LFM2.5-VL-1.6B"
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16"
)
processor = AutoProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/589 [00:00<?, ?it/s]

In [ ]:
url = "https://www.shutterstock.com/image-photo/vidhana-soudha-front-view-night-260nw-1146876431.jpg"
image = load_image(url)
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Which is th is building shown in the image?"},
        ],
    },
]

In [ ]:
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    tokenize=True,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=512)
output = processor.batch_decode(outputs, skip_special_tokens=True)[0]
lines = output.split('\n')
for line in lines:
    print(line)

user
Which is th is building shown in the image?
assistant
The building shown in the image is the Vidhana Soudha, which is the legislative assembly building of the Indian state of Karnataka. It is located in Bangalore, the capital city of Karnataka. The Vidhana Soudha is a prominent landmark in the city and serves as the seat of the Karnataka Legislative Assembly. The building is known for its distinctive architecture, which features a blend of traditional and modern design elements. It is a significant symbol of the state's political and administrative functions.


In [ ]:
follow_up = [    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": output},
        ]
    },
       {
        "role": "user",
        "content": [
            {"type": "text", "text": "What time in the day is this photo taken?"},
        ],
    }
]

conversation.extend(follow_up)

In [ ]:
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    tokenize=True,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=512)
full_conversation_output = processor.batch_decode(outputs, skip_special_tokens=True)[0]

# Extract only the last assistant's response
assistant_tag_start = "assistant\n"
last_assistant_idx = full_conversation_output.rfind(assistant_tag_start)

if last_assistant_idx != -1:
    second_response = full_conversation_output[last_assistant_idx + len(assistant_tag_start):].strip()
    print(second_response)
else:
    print("Could not find the second assistant response.")

This photo is taken at night. The image shows the Vidhana Soudha illuminated against the dark sky, with the lights of the surrounding area visible in the foreground. The long exposure used to capture the image creates a sense of motion and highlights the lights of the vehicles on the road, adding to the overall nighttime ambiance of the scene.
